## Oppgave 1. (10 poeng)
### Oppgave 1 b iii)

In [ ]:
#Genererer og plotter en trigonometrisk funksjon som tilfredsstiller de fem betingelsene 

import numpy as np
import matplotlib.pyplot as plt

vmax=25 #m/s
umax=0.2

#Vi prøver med en trigonometrisk funkjon, som går over en halv periode i intervallet mellom 0 og umax -> Bestemmer vinkelhastigheten B slik: pi/B=umax -> B=pi/umax
#Funksjonen skal spenne mellom 0 og vmax, og trenger derfor både amplitude og likevektslinje lik vmax/2

u = np.linspace (0,umax,1000)
v1 = vmax/2*np.cos(np.pi*u/umax)+vmax/2
plt.plot(u,v1)
plt.show()

In [ ]:
#Genererer og plotter en trigonometrisk funksjon som tilfredsstiller de fem betingelsene

# Vi legger opp til en funksjon av typen: v(u)=x-ln(u/umax+y)
# Den vil være monotont fallende i intervallet mellom u og umax.
# To likninger kan hjelpe oss med å bestemme verdiene for x og y
# (1) v(0)=vmax -> x-ln(y)=25 -> x=ln(y)+vmax
# (2) v(umax)=0 -> x-ln(1+y)=0 -> x=ln(1+y)
# Slår sammen (1) og (2) og får: ln(1+y)-ln(y)=vmax -> ln((1+y)/y)=vmax -> (1+y)/y=exp(vmax) -> 
y=1/(np.exp(vmax)-1)
x=np.log(y)+vmax

v2 = x-np.log(u/umax + y)
plt.plot(u,v2)
plt.show()


### e) Maksimal fluks 

In [4]:
#Finner den deriverte av fluksen

import sympy as sp

u,p,vmax,umax = sp.symbols("u, p, vmax, umax")
def v(u):
    return u*vmax*(1-(u/umax)**p)

pd=sp.diff(v(u), u)
print(pd)


-p*vmax*(u/umax)**p + vmax*(1 - (u/umax)**p)


## Oppgave 2. Rødt lys (15 poeng)


### a) Initialverdibetingelsen

In [ ]:
#Genererer og plotter funksjonen for initialbetingelsen u(x,0)

x=np.linspace(-1000,1000,2001)
umax=0.2
u0=np.full(2001,umax)
u0[1001:2001]=0

plt.plot(x,u0)
plt.show()

### c) Løs differensiallikningen

#### i) For $p=1$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# ------------------------------------------------------------
# Oppsett for figurer (nyttig i notater for å unngå "gamle" plott)
# ------------------------------------------------------------
plt.close("all")
plt.figure(figsize=(6, 4))

vmax=25
umax=0.2


# ------------------------------------------------------------
# Fluks for ligning: u_t + (f(u))_x = 0, der f(u) = u*vmax(1-(u/umax)**1)
# ------------------------------------------------------------
def flux(u):
    """Fluksen f(u) for likningen."""
    return u*vmax*(1-(u/umax)**1)

# ------------------------------------------------------------
# Lax–Friedrichs-metoden (1D, eksplisitt)
# ------------------------------------------------------------
# fluks: funksjon f(u)
# u: array med løsningen ved tid t (på romgitteret)
# dx: steglengde i rom
# dt: steglengde i tid
# t: nåværende tid (brukes av randbetingelsene)
# u_left(t): randverdi på venstre kant (x = a)
# u_right(t): randverdi på høyre kant (x = b)
def lax_friedrichs_step(fluks, u, dx, dt, t, u_left, u_right):
    """Utfører ett tidssteg med Lax–Friedrichs."""
    n = u.size
    u_next = np.zeros_like(u)

    # Indre punkter (i = 1, ..., n-2)
    # Formelen:
    # u_i^{n+1} = 1/2 (u_{i-1}^n + u_{i+1}^n) - (dt/(2dx)) (f(u_{i+1}^n) - f(u_{i-1}^n))
    for i in range(1, n - 1):
        u_next[i] = 0.5 * (u[i - 1] + u[i + 1]) - (dt / (2 * dx)) * (
            fluks(u[i + 1]) - fluks(u[i - 1])
        )

    # Randbetingelser (Dirichlet)
    u_next[0] = u_left(t)     # venstre rand
    u_next[-1] = u_right(t)   # høyre rand

    return u_next


# ------------------------------------------------------------
# Gitter i tid og rom
# ------------------------------------------------------------
T = 100.0          # sluttid
nt = 200         # antall tidspunkter
a, b = -1000.0, 1000.0 # romintervall
nx = 100         # antall rompunkter

t_grid = np.linspace(0.0, T, nt)
x_grid = np.linspace(a, b, nx)

dt = t_grid[1] - t_grid[0]
dx = x_grid[1] - x_grid[0]

# ------------------------------------------------------------
# Vindu 2: Sett initialbetingelse, løs med Lax–Friedrichs, og animer
# Forutsetter at vindu 1 allerede er kjørt (x_grid, t_grid, dx, dt, osv.)
# ------------------------------------------------------------

# Randbetingelser (Dirichlet)
def u_left(t):
    """Venstre randverdi u(a,t)."""
    return 0.0

def u_right(t):
    """Høyre randverdi u(b,t)."""
    return 0.0

# Initialbetingelse u(x,0)
u0=np.full(100,umax)
u0[51:100]=0

"""# CFL-sjekk (pedagogisk)
cfl = (dt / dx) * np.max(np.abs(u0))
print("max|u0| =", np.max(np.abs(u0)))
print("CFL ~ (dt/dx)*max|u0| =", cfl)"""

# Lagrer løsningen for alle tider (greit for animasjon)
u1 = np.zeros((t_grid.size, x_grid.size))
u1[0, :] = u0

# Tidsløkken
for n_step in range(1, t_grid.size):
    t_now = t_grid[n_step - 1]
    u1[n_step, :] = lax_friedrichs_step(
        flux, u1[n_step - 1, :], dx, dt, t_now, u_left, u_right
    )

# Diagnose: sjekk at løsningen faktisk endrer seg
print("Maks endring fra t0 til t1:", np.max(np.abs(u1[1, :] - u1[0, :])))


# ------------------------------------------------------------
# Animasjon
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 4))

# Faste aksegrenser (som i eksempelet)
ax.set_xlim(x_grid[0], x_grid[-1])
ax.set_ylim(np.min(u1), np.max(u1))

ax.set_xlabel("x")
ax.set_ylabel("u(x,t)")
ax.set_title("Numerisk løsning med Lax–Friedrichs-metoden, p=1, T=100")

# Rød stiplet: startprofil
ax.plot(x_grid, u0, color="red", linestyle="--", label="u(x,0) (startprofil)")

# Blå kurve: løsningen som oppdateres
line, = ax.plot(x_grid, u1[0, :], color="blue", label="u(x,t)")

ax.legend()

def animate(i):
    line.set_ydata(u1[i, :])
    return line,

ani = animation.FuncAnimation(
    fig,
    animate,
    frames=t_grid.size,
    interval=20,
    blit=True
)

HTML(ani.to_jshtml())
#plt.show()



#### ii) For $p=2$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# ------------------------------------------------------------
# Oppsett for figurer (nyttig i notater for å unngå "gamle" plott)
# ------------------------------------------------------------
plt.close("all")
plt.figure(figsize=(6, 4))

vmax=25
umax=0.2


# ------------------------------------------------------------
# Fluks for ligning: u_t + (f(u))_x = 0, der f(u) = u*vmax(1-(u/umax)**1)
# ------------------------------------------------------------
def flux(u):
    """Fluksen f(u) for likningen."""
    return u*vmax*(1-(u/umax)**2)

# ------------------------------------------------------------
# Lax–Friedrichs-metoden (1D, eksplisitt)
# ------------------------------------------------------------
# fluks: funksjon f(u)
# u: array med løsningen ved tid t (på romgitteret)
# dx: steglengde i rom
# dt: steglengde i tid
# t: nåværende tid (brukes av randbetingelsene)
# u_left(t): randverdi på venstre kant (x = a)
# u_right(t): randverdi på høyre kant (x = b)
def lax_friedrichs_step(fluks, u, dx, dt, t, u_left, u_right):
    """Utfører ett tidssteg med Lax–Friedrichs."""
    n = u.size
    u_next = np.zeros_like(u)

    # Indre punkter (i = 1, ..., n-2)
    # Formelen:
    # u_i^{n+1} = 1/2 (u_{i-1}^n + u_{i+1}^n) - (dt/(2dx)) (f(u_{i+1}^n) - f(u_{i-1}^n))
    for i in range(1, n - 1):
        u_next[i] = 0.5 * (u[i - 1] + u[i + 1]) - (dt / (2 * dx)) * (
            fluks(u[i + 1]) - fluks(u[i - 1])
        )

    # Randbetingelser (Dirichlet)
    u_next[0] = u_left(t)     # venstre rand
    u_next[-1] = u_right(t)   # høyre rand

    return u_next


# ------------------------------------------------------------
# Gitter i tid og rom
# ------------------------------------------------------------
T = 100.0          # sluttid
nt = 200         # antall tidspunkter
a, b = -1000.0, 1000.0 # romintervall
nx = 100         # antall rompunkter

t_grid = np.linspace(0.0, T, nt)
x_grid = np.linspace(a, b, nx)

dt = t_grid[1] - t_grid[0]
dx = x_grid[1] - x_grid[0]

# ------------------------------------------------------------
# Vindu 2: Sett initialbetingelse, løs med Lax–Friedrichs, og animer
# Forutsetter at vindu 1 allerede er kjørt (x_grid, t_grid, dx, dt, osv.)
# ------------------------------------------------------------

# Randbetingelser (Dirichlet)
def u_left(t):
    """Venstre randverdi u(a,t)."""
    return 0.0

def u_right(t):
    """Høyre randverdi u(b,t)."""
    return 0.0

# Initialbetingelse u(x,0)
u0=np.full(100,umax)
u0[51:100]=0

# CFL-sjekk (pedagogisk)
cfl = (dt / dx) * np.max(np.abs(u0))
print("max|u0| =", np.max(np.abs(u0)))
print("CFL ~ (dt/dx)*max|u0| =", cfl)

# Lagrer løsningen for alle tider (greit for animasjon)
u2 = np.zeros((t_grid.size, x_grid.size))
u2[0, :] = u0

# Tidsløkken
for n_step in range(1, t_grid.size):
    t_now = t_grid[n_step - 1]
    u2[n_step, :] = lax_friedrichs_step(
        flux, u2[n_step - 1, :], dx, dt, t_now, u_left, u_right
    )



# ------------------------------------------------------------
# Animasjon
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 4))

# Faste aksegrenser (som i eksempelet)
ax.set_xlim(x_grid[0], x_grid[-1])
ax.set_ylim(np.min(u2), np.max(u2))

ax.set_xlabel("x")
ax.set_ylabel("u(x,t)")
ax.set_title("Numerisk løsning med Lax–Friedrichs-metoden, p=2, T=100")

# Rød stiplet: startprofil
ax.plot(x_grid, u0, color="red", linestyle="--", label="u(x,0) (startprofil)")

# Blå kurve: løsningen som oppdateres
line, = ax.plot(x_grid, u2[0, :], color="blue", label="u(x,t)")

ax.legend()

def animate(i):
    line.set_ydata(u2[i, :])
    return line,

ani = animation.FuncAnimation(
    fig,
    animate,
    frames=t_grid.size,
    interval=20,
    blit=True
)

HTML(ani.to_jshtml())
#plt.show()

#### iii) For $p=5$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# ------------------------------------------------------------
# Oppsett for figurer
# ------------------------------------------------------------
plt.close("all")
plt.figure(figsize=(6, 4))

vmax = 25
umax = 0.2


# ------------------------------------------------------------
# Fluks
# ------------------------------------------------------------
def flux(u):
    """Fluksen f(u) for likningen."""
    return u * vmax * (1 - (u/umax)**5)


def flux_prime(u):
    return vmax * (1 - 6*(u/umax)**5)


# ------------------------------------------------------------
# Lax–Friedrichs-metoden
# ------------------------------------------------------------
def lax_friedrichs_step(fluks, u, dx, dt, t, u_left, u_right):
    """Utfører ett tidssteg med Lax–Friedrichs."""
    n = u.size
    u_next = np.zeros_like(u)

    # Indre punkter
    for i in range(1, n - 1):
        u_next[i] = 0.5 * (u[i - 1] + u[i + 1]) - (dt / (2 * dx)) * (
            fluks(u[i + 1]) - fluks(u[i - 1])
        )

    # Randbetingelser
    u_next[0] = u_left(t)
    u_next[-1] = u_right(t)

    return u_next


# ------------------------------------------------------------
# Gitter i tid og rom
# ------------------------------------------------------------
T = 100.0
nt = 1000

a, b = -1000.0, 1000.0
nx = 100

t_grid = np.linspace(0.0, T, nt)
x_grid = np.linspace(a, b, nx)

dt = t_grid[1] - t_grid[0]
dx = x_grid[1] - x_grid[0]


# ------------------------------------------------------------
# Randbetingelser
# ------------------------------------------------------------
def u_left(t):
    return 0.0


def u_right(t):
    return 0.0


# ------------------------------------------------------------
# Initialbetingelse
# ------------------------------------------------------------
u0 = np.full(nx, umax)
u0[int(nx/2):] = 0


# ------------------------------------------------------------
# CFL-sjekk
# ------------------------------------------------------------
cfl = (dt/dx) * np.max(np.abs(flux_prime(u0)))

print("max |f'(u)| =", np.max(np.abs(flux_prime(u0))))
print("CFL =", cfl)


# ------------------------------------------------------------
# Lagrer løsningen
# ------------------------------------------------------------
u5 = np.zeros((t_grid.size, x_grid.size))
u5[0, :] = u0


# ------------------------------------------------------------
# Tidsløkken
# ------------------------------------------------------------
for n_step in range(1, t_grid.size):

    t_now = t_grid[n_step - 1]

    u5[n_step, :] = lax_friedrichs_step(
        flux,
        u5[n_step - 1, :],
        dx,
        dt,
        t_now,
        u_left,
        u_right
    )


# ------------------------------------------------------------
# Animasjon
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 4))

ax.set_xlim(x_grid[0], x_grid[-1])
ax.set_ylim(np.min(u5), np.max(u5))

ax.set_xlabel("x")
ax.set_ylabel("u(x,t)")
ax.set_title("Numerisk løsning med Lax–Friedrichs-metoden, p=5, T=100")

# Startprofil
ax.plot(x_grid, u0, color="red", linestyle="--", label="u(x,0)")

# Løsning
line, = ax.plot(x_grid, u5[0, :], color="blue", label="u(x,t)")

ax.legend()


def animate(i):
    line.set_ydata(u5[i, :])
    return line,


ani = animation.FuncAnimation(
    fig,
    animate,
    frames=t_grid.size,
    interval=20,
    blit=True
)

HTML(ani.to_jshtml())

### d) Bevegelsen til en enkelt bil

#### i) For $p=1$

In [ ]:

# Fartsfunksjonen når p=1
def fart(u):
    return vmax * (1 - (u/umax)**1)

T = 100.0          # sluttid
nt = 200         # antall tidspunkter
a, b = -1000.0, 1000.0 # romintervall
nx = 100         # antall rompunkter

t_grid = np.linspace(0.0, T, nt)
x_grid = np.linspace(a, b, nx)

dt = t_grid[1] - t_grid[0]
dx = x_grid[1] - x_grid[0]

# lager plass til verdier
P = np.zeros(t_grid.size)
V = np.zeros(t_grid.size)

# Initialbetingelser
P[0] = -1000.0
V[0] = 0.0


for n in range(t_grid.size - 1):
    u_P = np.interp(P[n], x_grid, u1[n, :])
    V[n] = fart(u_P)
    P[n+1] = P[n] + dt*V[n]

#siste V-verdi
u_P = np.interp(P[-1], x_grid, u1[-1, :])
V[-1] = fart(u_P)

plt.subplot(1,2,1)
plt.plot(t_grid, P)
plt.xlabel("t")
plt.ylabel("P(t)")
plt.title("Posisjon, p=1")

plt.subplot(1,2,2)
plt.plot(t_grid, V)
plt.xlabel("t")
plt.ylabel("V(t)")
plt.title("Fart, p=1")

plt.tight_layout()
plt.show()



#### i) For $p=2$

In [ ]:
# Fartsfunksjonen når p=2
def fart(u):
    return vmax * (1 - (u/umax)**2)

# lager plass til verdier
P = np.zeros(t_grid.size)
V = np.zeros(t_grid.size)

# Initialbetingelser
P[0] = -1000.0
V[0] = 0.0

for n in range(t_grid.size - 1):
    u_P = np.interp(P[n], x_grid, u2[n, :])
    V[n] = fart(u_P)
    P[n+1] = P[n] + dt*V[n]

#siste V-verdi
u_P = np.interp(P[-1], x_grid, u2[-1, :])
V[-1] = fart(u_P)

plt.subplot(1,2,1)
plt.plot(t_grid, P)
plt.xlabel("t")
plt.ylabel("P(t)")
plt.title("Posisjon, p=2")

plt.subplot(1,2,2)
plt.plot(t_grid, V)
plt.xlabel("t")
plt.ylabel("V(t)")
plt.title("Fart, p=2")

plt.tight_layout()
plt.show()


#### i) For $p=5$

In [ ]:
# Fartsfunksjonen når p=5
def fart(u):
    return vmax * (1 - (u/umax)**5)

T = 100.0
nt = 1000

a, b = -1000.0, 1000.0
nx = 100

t_grid = np.linspace(0.0, T, nt)
x_grid = np.linspace(a, b, nx)

dt = t_grid[1] - t_grid[0]
dx = x_grid[1] - x_grid[0]

# lager plass til verdier
P = np.zeros(t_grid.size)
V = np.zeros(t_grid.size)

# Initialbetingelser
P[0] = -1000.0
V[0] = 0.0

for n in range(t_grid.size - 1):
    u_P = np.interp(P[n], x_grid, u5[n, :])
    V[n] = fart(u_P)
    P[n+1] = P[n] + dt*V[n]

#siste V-verdi
u_P = np.interp(P[-1], x_grid, u5[-1, :])
V[-1] = fart(u_P)

plt.subplot(1,2,1)
plt.plot(t_grid, P)
plt.xlabel("t")
plt.ylabel("P(t)")
plt.title("Posisjon, p=5")

plt.subplot(1,2,2)
plt.plot(t_grid, V)
plt.xlabel("t")
plt.ylabel("V(t)")
plt.title("Fart, p=5")

plt.tight_layout()
plt.show()

### e) Hastigheten til en kø

#### i) For $p=1$

In [ ]:

#Bestemmer køfront for p=1

T = 100.0          # sluttid
nt = 200         # antall tidspunkter
a, b = -1000.0, 1000.0 # romintervall
nx = 100         # antall rompunkter

t_grid = np.linspace(0.0, T, nt)
x_grid = np.linspace(a, b, nx)

dt = t_grid[1] - t_grid[0]
dx = x_grid[1] - x_grid[0]

K = np.zeros(t_grid.size)
epsilon = 1e-3

for n in range(t_grid.size):

    u_now = u1[n, :]

    # søk fra høyre mot venstre etter siste punkt ~ umax
    idx = np.where(u_now > umax - epsilon)[0]

    if len(idx) > 0 and idx[-1] < nx-1:

        i = idx[-1]

        # lineær interpolasjon
        x1 = x_grid[i]
        x2 = x_grid[i+1]

        u1_val = u_now[i]
        u2_val = u_now[i+1]

        K[n] = x1 + (umax - u1_val) * (x2 - x1) / (u2_val - u1_val)

    else:
        K[n] = x_grid[0]


plt.figure()

plt.plot(t_grid, K)

plt.xlabel("t")
plt.ylabel("K(t)")
plt.title("Køfrontens posisjon, p=1")

plt.show()

terskel=-1000
n=0
while K[n]> terskel:
    n=n+1
print(n)
tid = n*dt
print(f"Etter {tid:.2f} sekunder er alle bilene i bevegelse når p=1")


#### i) For $p=2$

In [ ]:

#Bestemmer køfront for p=2

K = np.zeros(t_grid.size)
epsilon = 1e-3

for n in range(t_grid.size):

    u_now = u2[n, :]

    # søk fra høyre mot venstre etter siste punkt ~ umax
    idx = np.where(u_now > umax - epsilon)[0]

    if len(idx) > 0 and idx[-1] < nx-1:

        i = idx[-1]

        # lineær interpolasjon
        x1 = x_grid[i]
        x2 = x_grid[i+1]

        u1_val = u_now[i]
        u2_val = u_now[i+1]

        K[n] = x1 + (umax - u1_val) * (x2 - x1) / (u2_val - u1_val)

    else:
        K[n] = x_grid[0]


plt.figure()

plt.plot(t_grid, K)

plt.xlabel("t")
plt.ylabel("K(t)")
plt.title("Køfrontens posisjon, p=2")

plt.show()

terskel=-1000
n=0
while K[n]> terskel:
    n=n+1
print(n)
tid = n*dt
print(f"Etter {tid:.2f} sekunder er alle bilene i bevegelse når p=2")





#### i) For $p=5$

In [ ]:

#Bestemmer køfront for p=5

T = 100.0
nt = 1000

a, b = -1000.0, 1000.0
nx = 100

t_grid = np.linspace(0.0, T, nt)
x_grid = np.linspace(a, b, nx)

dt = t_grid[1] - t_grid[0]
dx = x_grid[1] - x_grid[0]

K = np.zeros(t_grid.size)
epsilon = 1e-3

for n in range(t_grid.size):

    u_now = u5[n, :]

    # søk fra høyre mot venstre etter siste punkt ~ umax
    idx = np.where(u_now > umax - epsilon)[0]

    if len(idx) > 0 and idx[-1] < nx-1:

        i = idx[-1]

        # lineær interpolasjon
        x1 = x_grid[i]
        x2 = x_grid[i+1]

        u1_val = u_now[i]
        u2_val = u_now[i+1]

        K[n] = x1 + (umax - u1_val) * (x2 - x1) / (u2_val - u1_val)

    else:
        K[n] = x_grid[0]


plt.figure()

plt.plot(t_grid, K)

plt.xlabel("t")
plt.ylabel("K(t)")
plt.title("Køfrontens posisjon, p=5")

plt.show()

terskel=-1000
n=0
while K[n]> terskel:
    n=n+1
print(n)
tid = n*dt
print(f"Etter {tid:.2f} sekunder er alle bilene i bevegelse når p=5")


### f) Valg av parameter $p$

In [ ]:
#Gerererer figur som illustrerer fartsfunksjonen for ulike verdier av p

import numpy as np
import matplotlib.pyplot as plt

uu = np.linspace(0, 0.2, 200)

vv5 = vmax*(1-(uu/umax)**5)
vv2 = vmax*(1-(uu/umax)**2)
vv1 = vmax*(1-(uu/umax)**1)

plt.plot(uu, vv1, label="p=1")
plt.plot(uu, vv2, label="p=2")
plt.plot(uu, vv5, label="p=5")

plt.xlabel("u")
plt.ylabel("v(u)")
plt.title("Fartsfunksjoner for ulike p")
plt.legend()
plt.show()